# Experiment Queue Runner

Resumable experiment runner for Google Colab.  
After every completed experiment the notebook writes `checkpoint_<n>.md` and syncs `registry.json` to Google Drive.  
On the next Colab session it finds the latest checkpoint and picks up at `n+1`.

**Sections**
- **§1 Main** — 6 datasets × 4 horizons × 7 models × 3 seeds = 504 runs
- **§3 Data fraction** — 4 fractions × 7 models × 3 seeds = 84 runs (ETTh1, pred_len=96)
- **Total: 588 runs**

Charts and tables aggregate metrics across seeds (mean ± std).

**Workflow**
1. Upload project files to Colab (e.g. `/content/multimodality`), set `PROJECT_ROOT` in cell 2
2. Run cells 1–8 (setup, no side-effects)
3. Run cell 9 (run loop) — resumes automatically on reconnect
4. Run cells 10–13 at any time to inspect partial results

In [ ]:
# ── Cell 2: Google Drive mount + project root ─────────────────────────────────
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = Path('/content/drive/MyDrive/multimodality_experiments')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Set this to the folder where you uploaded the project files ───────────────
PROJECT_ROOT = Path('/content/multimodality')  # <-- edit if different
os.chdir(PROJECT_ROOT)

print(f'Working directory : {PROJECT_ROOT}')
print(f'Drive output dir  : {DRIVE_DIR}')

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.0.0', 'torchvision>=0.15.0',
    'numpy>=1.23.0', 'pandas>=1.5.0', 'scikit-learn>=1.2.0',
    'PyYAML>=6.0', 'einops>=0.6.0',
    'transformers>=4.30.0',
    'scipy>=1.10.0', 'tqdm>=4.64.0', 'matplotlib>=3.7.0',
])
print('Dependencies ready.')

In [ ]:
# ── Cell 4: GPU check + project imports ──────────────────────────────────────
import json, shutil, re
from datetime import datetime, timezone
import torch

if torch.cuda.is_available():
    device_info = f'CUDA — {torch.cuda.get_device_name(0)}'
elif torch.backends.mps.is_available():
    device_info = 'Apple MPS'
else:
    device_info = 'CPU  (text models require GPU and will be skipped)'

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device_info}')

from run_experiment import run
from compare_results import load_registry, keep_latest, print_table

In [ ]:
# ── Cell 5: Dataset catalogue ─────────────────────────────────────────────────
DATASETS = {
    'ETTh1':       dict(dataset='ETTh1',  root_path='./dataset/ETT-small/', data_path='ETTh1.csv',       freq='h', enc_in=7),
    'ETTh2':       dict(dataset='ETTh2',  root_path='./dataset/ETT-small/', data_path='ETTh2.csv',       freq='h', enc_in=7),
    'ETTm1':       dict(dataset='ETTm1',  root_path='./dataset/ETT-small/', data_path='ETTm1.csv',       freq='t', enc_in=7),
    'ETTm2':       dict(dataset='ETTm2',  root_path='./dataset/ETT-small/', data_path='ETTm2.csv',       freq='t', enc_in=7),
    'Weather':     dict(dataset='custom', root_path='./dataset/weather/',    data_path='weather.csv',     freq='h', enc_in=21),
    # 'Electricity': dict(dataset='custom', root_path='./dataset/electricity/',data_path='electricity.csv', freq='h', enc_in=321),
}

In [ ]:
# ── Cell 6: Constants ─────────────────────────────────────────────────────────
CONFIGS = [
    ('01_dlinear',         'dlinear'),
    ('02_patchtst',        'patchtst'),
    ('03_bert_forecaster', 'bert_forecaster'),
    ('04_late_fusion',     'late_fusion'),
    ('05_gated_fusion',    'gated_fusion'),
    ('06_film_fusion',     'film_fusion'),
    ('07_ensemble_fusion', 'ensemble_fusion'),
]

SEEDS           = [2024, 2025, 2026]
HORIZONS        = [192, 336]#[96, 192, 336, 720]
TRAIN_FRACTIONS = [1.0, 0.5, 0.1]
FRAC_DATASET    = 'ETTh1'
FRAC_HORIZON    = 96

DS_SLUG = {
    'ETTh1':       'etth1',
    'ETTh2':       'etth2',
    'ETTm1':       'ettm1',
    'ETTm2':       'ettm2',
    'Weather':     'weather',
}

In [ ]:
# ── Cell 7: Build experiment queue ────────────────────────────────────────────
queue = []

# §1 Main: 6 datasets × 4 horizons × 7 models × 3 seeds = 504 runs
for ds_key, ds in DATASETS.items():
    for pred_len in HORIZONS:
        for cfg_prefix, label in CONFIGS:
            for seed in SEEDS:
                slug = DS_SLUG[ds_key]
                name = f'{label}_{slug}_pred{pred_len}_s{seed}'
                queue.append(dict(
                    section     = 1,
                    config_path = f'experiments/configs/{cfg_prefix}_{slug}.yaml',
                    name        = name,
                    overrides   = [
                        f'name={name}',
                        f'data.dataset={ds["dataset"]}',
                        f'data.root_path={ds["root_path"]}',
                        f'data.data_path={ds["data_path"]}',
                        f'data.freq={ds["freq"]}',
                        f'model.enc_in={ds["enc_in"]}',
                        f'model.pred_len={pred_len}',
                        f'training.seed={seed}',
                    ],
                    ds_key   = ds_key,
                    pred_len = pred_len,
                    label    = label,
                    seed     = seed,
                ))

# §3 Data fraction: 4 fractions × 7 models × 3 seeds = 84 runs (fixed ETTh1, pred_len=96)
_ds   = DATASETS[FRAC_DATASET]
_slug = DS_SLUG[FRAC_DATASET]
for frac in TRAIN_FRACTIONS:
    frac_tag = f'frac{int(frac * 100)}'
    for cfg_prefix, label in CONFIGS:
        for seed in SEEDS:
            name = f'{label}_{_slug}_pred{FRAC_HORIZON}_{frac_tag}_s{seed}'
            queue.append(dict(
                section     = 3,
                config_path = f'experiments/configs/{cfg_prefix}_{_slug}.yaml',
                name        = name,
                overrides   = [
                    f'name={name}',
                    f'data.dataset={_ds["dataset"]}',
                    f'data.root_path={_ds["root_path"]}',
                    f'data.data_path={_ds["data_path"]}',
                    f'data.freq={_ds["freq"]}',
                    f'model.enc_in={_ds["enc_in"]}',
                    f'model.pred_len={FRAC_HORIZON}',
                    f'training.train_fraction={frac}',
                    f'training.seed={seed}',
                ],
                ds_key   = FRAC_DATASET,
                pred_len = FRAC_HORIZON,
                label    = label,
                frac     = frac,
                seed     = seed,
            ))

# Stamp indices
for i, job in enumerate(queue):
    job['index'] = i

TOTAL = len(queue)
print(f'Queue built: {TOTAL} experiments')
print(f'  §1 main          : {sum(j["section"] == 1 for j in queue)}')
print(f'  §3 data fraction : {sum(j["section"] == 3 for j in queue)}')

In [ ]:
# ── Cell 8: Checkpoint utilities + seed aggregation ───────────────────────────
import numpy as np
from collections import defaultdict

CHECKPOINT_PREFIX = 'checkpoint_'


def write_checkpoint(job: dict, metrics: dict, total: int):
    """Write checkpoint_{n}.md to Drive after a successful run."""
    n   = job['index']
    ts  = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
    mae = metrics.get('mae', float('nan'))
    mse = metrics.get('mse', float('nan'))
    content = (
        f'# Checkpoint {n}\n\n'
        f'- **Experiment index**: {n}\n'
        f'- **Name**: {job["name"]}\n'
        f'- **Section**: §{job["section"]}\n'
        f'- **Config**: {job["config_path"]}\n'
        f'- **Seed**: {job.get("seed", "n/a")}\n'
        f'- **Timestamp**: {ts}\n'
        f'- **MAE**: {mae:.6f}\n'
        f'- **MSE**: {mse:.6f}\n'
        f'- **Progress**: {n + 1} / {total}\n'
    )
    (DRIVE_DIR / f'{CHECKPOINT_PREFIX}{n}.md').write_text(content)


def find_resume_index() -> int:
    """Return the first experiment index that has no checkpoint file.
    Scans DRIVE_DIR for checkpoint_*.md and finds the first gap in 0..TOTAL-1.
    Returns TOTAL if all experiments are done."""
    pattern = re.compile(rf'^{re.escape(CHECKPOINT_PREFIX)}(\d+)\.md$')
    done = {
        int(m.group(1))
        for p in DRIVE_DIR.iterdir()
        if (m := pattern.match(p.name))
    }
    for i in range(TOTAL):
        if i not in done:
            return i
    return TOTAL


def sync_registry_to_drive():
    """Copy local registry.json → Drive after every experiment."""
    src = PROJECT_ROOT / 'experiments' / 'registry.json'
    if src.exists():
        shutil.copy2(src, DRIVE_DIR / 'registry.json')


def sync_plot_to_drive(png_path: Path):
    shutil.copy2(png_path, DRIVE_DIR / png_path.name)


def restore_registry_from_drive():
    """On a fresh Colab session, restore prior results from Drive so
    load_registry() returns all previously completed experiments."""
    drive_reg = DRIVE_DIR / 'registry.json'
    local_reg = PROJECT_ROOT / 'experiments' / 'registry.json'
    if drive_reg.exists():
        shutil.copy2(drive_reg, local_reg)
        data = json.loads(local_reg.read_text())
        print(f'Registry restored from Drive ({len(data)} experiments).')
    else:
        local_reg.parent.mkdir(parents=True, exist_ok=True)
        local_reg.write_text('[]')
        print('No Drive registry found — starting fresh.')


# ── Seed aggregation helpers ──────────────────────────────────────────────────

def strip_seed(name: str, seeds: list) -> str:
    """Remove _s{seed} suffix to get the base experiment name."""
    for s in seeds:
        suffix = f'_s{s}'
        if name.endswith(suffix):
            return name[:-len(suffix)]
    return name


def aggregate_seeds(records: list, seeds: list) -> list:
    """Group records by base name (without _s{seed}), return mean ± std per metric."""
    groups: dict[str, list] = defaultdict(list)
    for r in records:
        base = strip_seed(r['name'], seeds)
        groups[base].append(r)

    agg = []
    for base_name, recs in groups.items():
        metrics_agg = {}
        for key in ['mae', 'mse', 'rmse']:
            vals = [r['metrics'].get(key, float('nan')) for r in recs]
            vals = [v for v in vals if not np.isnan(v)]
            if vals:
                metrics_agg[key]          = float(np.mean(vals))
                metrics_agg[f'{key}_std'] = float(np.std(vals))
            else:
                metrics_agg[key]          = float('nan')
                metrics_agg[f'{key}_std'] = float('nan')
        ref = recs[0]
        agg.append({
            'name':           base_name,
            'model':          ref.get('model', ''),
            'dataset':        ref.get('dataset', ''),
            'pred_len':       ref.get('pred_len'),
            'train_fraction': ref.get('train_fraction', 1.0),
            'n_seeds':        len(recs),
            'metrics':        metrics_agg,
        })
    return agg


def print_agg_table(agg_records: list, sort_by: str = 'mse'):
    """Print aggregated results table showing mean ± std."""
    if not agg_records:
        print('No results yet.')
        return
    if sort_by:
        agg_records = sorted(agg_records, key=lambda r: r['metrics'].get(sort_by, float('inf')))

    name_w  = max(len(r['name'])  for r in agg_records) + 2
    model_w = max(len(r['model']) for r in agg_records) + 2
    header = (
        f"{'name':<{name_w}} {'model':<{model_w}} "
        f"{'pred_len':>8}  {'frac':>5}  {'n':>2}  "
        f"{'MAE (mean±std)':>20}  {'MSE (mean±std)':>20}"
    )
    sep = '-' * len(header)
    print(sep)
    print(header)
    print(sep)
    for r in agg_records:
        m    = r['metrics']
        frac = r.get('train_fraction', 1.0)
        mae_str = f"{m.get('mae', float('nan')):.4f}±{m.get('mae_std', float('nan')):.4f}"
        mse_str = f"{m.get('mse', float('nan')):.4f}±{m.get('mse_std', float('nan')):.4f}"
        print(
            f"{r['name']:<{name_w}} {r['model']:<{model_w}} "
            f"{r['pred_len']:>8}  {frac:>4.0%}  {r['n_seeds']:>2}  "
            f"{mae_str:>20}  {mse_str:>20}"
        )
    print(sep)
    print(f'{len(agg_records)} experiment(s)')


print('Utilities ready.')

In [ ]:
# ── Cell 9: Run loop ──────────────────────────────────────────────────────────
# Re-running this cell after a Colab disconnect resumes from the first missing checkpoint.

restore_registry_from_drive()

start_index = find_resume_index()
print(f'Total experiments : {TOTAL}')

if start_index >= TOTAL:
    print('All experiments complete.')
else:
    print(f'Resuming from index {start_index}  ({TOTAL - start_index} remaining)')

for job in queue[start_index:]:
    n = job['index']
    print(f'\n{"="*70}')
    print(f'[{n + 1}/{TOTAL}]  §{job["section"]}  {job["name"]}')
    print(f'{"="*70}')

    try:
        metrics = run(
            config_path   = job['config_path'],
            name_override = job['name'],
            overrides     = job['overrides'],
        )
        write_checkpoint(job, metrics, TOTAL)
        sync_registry_to_drive()
        print(f'  checkpoint_{n}.md → Drive')
        print(f'  MAE={metrics.get("mae", float("nan")):.4f}  '
              f'MSE={metrics.get("mse", float("nan")):.4f}')

    except SystemExit:
        # GPU unavailable — no checkpoint written, so this job will be retried
        # on the next session (which may have a GPU).
        print('  SKIPPED (GPU required but not available) — will retry next session')
        break

    except Exception as exc:
        # No checkpoint written → job will be retried on next resume.
        print(f'  ERROR: {exc}')
        print('  Stopping — re-run this cell to retry from this job.')
        break

print('\nRun loop finished.')

---
## Visualisations
Run the cells below at any time — they work with partial results.

In [ ]:
# ── Cell 10: §1 bar charts (mean MAE + MSE ± std per dataset × horizon) ───────
import matplotlib.pyplot as plt
import numpy as np

all_records = keep_latest(load_registry())

for ds_key in DATASETS:
    slug   = DS_SLUG[ds_key]
    for pred_len in HORIZONS:
        # Filter to §1 records for this (dataset, horizon): base name ends with _pred{pred_len}
        suffix = f'_pred{pred_len}'
        recs = [
            r for r in all_records
            if strip_seed(r['name'], SEEDS).endswith(suffix)
            and f'_{slug}_' in r['name']
        ]
        if not recs:
            continue

        agg = aggregate_seeds(recs, SEEDS)
        agg_sorted = sorted(agg, key=lambda r: r['metrics'].get('mse', float('inf')))
        labels   = [r['name'].replace(f'_{slug}{suffix}', '') for r in agg_sorted]
        maes     = [r['metrics'].get('mae',     float('nan')) for r in agg_sorted]
        mae_stds = [r['metrics'].get('mae_std', 0.0)          for r in agg_sorted]
        mses     = [r['metrics'].get('mse',     float('nan')) for r in agg_sorted]
        mse_stds = [r['metrics'].get('mse_std', 0.0)          for r in agg_sorted]

        x = np.arange(len(labels))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        fig.suptitle(f'§1  {ds_key} | pred_len={pred_len}  (mean ± std, n={len(SEEDS)} seeds)', fontsize=12)

        ax1.bar(x, maes, yerr=mae_stds, capsize=4, color='steelblue', error_kw=dict(elinewidth=1))
        ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=30, ha='right')
        ax1.set_ylabel('MAE'); ax1.set_title('MAE')

        ax2.bar(x, mses, yerr=mse_stds, capsize=4, color='coral', error_kw=dict(elinewidth=1))
        ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=30, ha='right')
        ax2.set_ylabel('MSE'); ax2.set_title('MSE')

        plt.tight_layout()
        png_path = PROJECT_ROOT / 'experiments' / f'main_{slug}_pred{pred_len}.png'
        plt.savefig(png_path, dpi=150, bbox_inches='tight')
        sync_plot_to_drive(png_path)
        plt.show()
        plt.close()

In [ ]:
# ── Cell 11: §3 line chart (mean MSE ± std vs train fraction per model) ───────
import matplotlib.pyplot as plt

all_records = keep_latest(load_registry())
_slug       = DS_SLUG[FRAC_DATASET]

# Collect {base_label: {frac: (mean_mse, std_mse)}}
frac_data: dict[str, dict[float, tuple]] = {}
for frac in TRAIN_FRACTIONS:
    frac_tag = f'frac{int(frac * 100)}'
    for _, label in CONFIGS:
        base_name = f'{label}_{_slug}_pred{FRAC_HORIZON}_{frac_tag}'
        recs = [
            r for r in all_records
            if strip_seed(r['name'], SEEDS) == base_name
        ]
        if recs:
            vals = [r['metrics'].get('mse', float('nan')) for r in recs]
            vals = [v for v in vals if not __import__('math').isnan(v)]
            if vals:
                import numpy as _np
                frac_data.setdefault(label, {})[frac] = (float(_np.mean(vals)), float(_np.std(vals)))

if not frac_data:
    print('No §3 results yet.')
else:
    import numpy as np
    fracs_sorted = sorted(TRAIN_FRACTIONS, reverse=True)  # 100% → 5%
    x_labels     = [f'{int(f * 100)}%' for f in fracs_sorted]
    x            = np.arange(len(x_labels))

    fig, ax = plt.subplots(figsize=(10, 5))
    for label, fd in sorted(frac_data.items()):
        means = [fd[f][0] if f in fd else float('nan') for f in fracs_sorted]
        stds  = [fd[f][1] if f in fd else 0.0          for f in fracs_sorted]
        ax.plot(x_labels, means, marker='o', label=label)
        ax.fill_between(
            x_labels,
            [m - s for m, s in zip(means, stds)],
            [m + s for m, s in zip(means, stds)],
            alpha=0.15,
        )

    ax.set_xlabel('Training data fraction (most recent windows)')
    ax.set_ylabel('MSE (mean ± std)')
    ax.set_title(f'§3  MSE vs Train Fraction — {FRAC_DATASET} pred_len={FRAC_HORIZON}  (n={len(SEEDS)} seeds)')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    png_path = PROJECT_ROOT / 'experiments' / f'fraction_mse_{_slug}.png'
    plt.savefig(png_path, dpi=150, bbox_inches='tight')
    sync_plot_to_drive(png_path)
    plt.show()
    plt.close()

---
## Summary Tables

In [ ]:
# ── Cell 12: §1 summary tables (mean ± std, one per dataset × horizon) ────────
all_records = keep_latest(load_registry())

print('=' * 80)
print('§1  MAIN RESULTS  (mean ± std across seeds)')
print('=' * 80)

for ds_key in DATASETS:
    slug = DS_SLUG[ds_key]
    for pred_len in HORIZONS:
        suffix = f'_pred{pred_len}'
        recs = [
            r for r in all_records
            if strip_seed(r['name'], SEEDS).endswith(suffix)
            and f'_{slug}_' in r['name']
        ]
        if not recs:
            continue
        agg = aggregate_seeds(recs, SEEDS)
        print(f'\n=== {ds_key} | pred_len={pred_len} ===')
        print_agg_table(agg, sort_by='mse')

In [ ]:
# ── Cell 13: §3 summary tables (mean ± std, one per fraction) ─────────────────
all_records = keep_latest(load_registry())
_slug       = DS_SLUG[FRAC_DATASET]

print('=' * 80)
print('§3  DATA FRACTION RESULTS  (mean ± std across seeds)')
print('=' * 80)

for frac in TRAIN_FRACTIONS:
    frac_tag = f'frac{int(frac * 100)}'
    recs = [
        r for r in all_records
        if strip_seed(r['name'], SEEDS).endswith(f'_{frac_tag}')
        and f'_{_slug}_' in r['name']
    ]
    if not recs:
        continue
    agg = aggregate_seeds(recs, SEEDS)
    print(f'\n=== {FRAC_DATASET} pred_len={FRAC_HORIZON} | train={frac:.0%} ===')
    print_agg_table(agg, sort_by='mse')